## About dataset

#### Content
The water_potability.csv file contains water quality metrics for 3276 different water bodies.

**1. pH value:**
PH is an important parameter in evaluating the acid–base balance of water. It is also the indicator of acidic or alkaline condition of water status.
WHO has recommended maximum permissible limit of pH from 6.5 to 8.5. The current investigation ranges were 6.52–6.83 which are in the range of WHO standards.

**2. Hardness:**
Hardness is mainly caused by calcium and magnesium salts. These salts are dissolved from geologic deposits through which water travels. The length of time water is in contact with hardness producing material helps determine how much hardness there is in raw water. Hardness was originally defined as the capacity of water to precipitate soap caused by Calcium and Magnesium.

**3. Solids (Total dissolved solids - TDS):**
Water has the ability to dissolve a wide range of inorganic and some organic minerals or salts such as potassium, calcium, sodium, bicarbonates, chlorides, magnesium, sulfates etc. These minerals produced un-wanted taste and diluted color in appearance of water. This is the important parameter for the use of water. The water with high TDS value indicates that water is highly mineralized. Desirable limit for TDS is 500 mg/l and maximum limit is 1000 mg/l which prescribed for drinking purpose.

**4. Chloramines:**
Chlorine and chloramine are the major disinfectants used in public water systems. Chloramines are most commonly formed when ammonia is added to chlorine to treat drinking water. Chlorine levels up to 4 milligrams per liter (mg/L or 4 parts per million (ppm)) are considered safe in drinking water.

**5. Sulfate:**
Sulfates are naturally occurring substances that are found in minerals, soil, and rocks. They are present in ambient air, groundwater, plants, and food. The principal commercial use of sulfate is in the chemical industry. Sulfate concentration in seawater is about 2,700 milligrams per liter (mg/L). It ranges from 3 to 30 mg/L in most freshwater supplies, although much higher concentrations (1000 mg/L) are found in some geographic locations.

**6. Conductivity:**
Pure water is not a good conductor of electric current rather’s a good insulator. Increase in ions concentration enhances the electrical conductivity of water. Generally, the amount of dissolved solids in water determines the electrical conductivity. Electrical conductivity (EC) actually measures the ionic process of a solution that enables it to transmit current. According to WHO standards, EC value should not exceeded 400 μS/cm.

**7. Organic_carbon:**
Total Organic Carbon (TOC) in source waters comes from decaying natural organic matter (NOM) as well as synthetic sources. TOC is a measure of the total amount of carbon in organic compounds in pure water. According to US EPA < 2 mg/L as TOC in treated / drinking water, and < 4 mg/Lit in source water which is use for treatment.

**8. Trihalomethanes:**
THMs are chemicals which may be found in water treated with chlorine. The concentration of THMs in drinking water varies according to the level of organic material in the water, the amount of chlorine required to treat the water, and the temperature of the water that is being treated. THM levels up to 80 ppm is considered safe in drinking water.

**9. Turbidity:**
The turbidity of water depends on the quantity of solid matter present in the suspended state. It is a measure of light emitting properties of water and the test is used to indicate the quality of waste discharge with respect to colloidal matter. The mean turbidity value obtained for Wondo Genet Campus (0.98 NTU) is lower than the WHO recommended value of 5.00 NTU.

**10. Potability:**
Indicates if water is safe for human consumption where 1 means Potable and 0 means Not potable.

In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score
import wandb
import yaml

In [4]:
def load_and_preprocess_data(file_path):
    print("Loading and preprocessing data...")
    df = pd.read_csv(file_path)
    df['ph'].fillna(df['ph'].mean(), inplace=True)
    df['Sulfate'].fillna(df['Sulfate'].mean(), inplace=True)
    df['Trihalomethanes'].fillna(df['Trihalomethanes'].mean(), inplace=True)

    X = df.drop('Potability', axis=1)
    y = df['Potability']
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    print("Data ready.")
    return X_train_scaled, X_test_scaled, y_train, y_test

In [7]:
def train():
    with wandb.init() as run:
        config = wandb.config
        print(f"Starting a new run for model: {config.model_name} with parameters:", dict(config))
        if config.model_name == 'svm':
            if config.kernel == 'poly':
                model = SVC(
                    C=config.C,
                    kernel=config.kernel,
                    gamma=config.gamma,
                    degree=config.degree,
                    random_state=42
                )
            else:
                model = SVC(
                    C=config.C,
                    kernel=config.kernel,
                    gamma=config.gamma,
                    random_state=42
                )
        elif config.model_name == 'random_forest':
            model = RandomForestClassifier(
                n_estimators=config.n_estimators_rf,
                max_depth=config.max_depth_rf,
                min_samples_leaf=config.min_samples_leaf,
                criterion=config.criterion,
                random_state=42
            )
        elif config.model_name == 'xgboost':
            model = XGBClassifier(
                n_estimators=config.n_estimators_xgb,
                max_depth=config.max_depth_xgb,
                learning_rate=config.learning_rate,
                subsample=config.subsample,
                use_label_encoder=False,
                eval_metric='logloss',
                random_state=42
            )
        elif config.model_name == 'knn':
            model = KNeighborsClassifier(
                n_neighbors=config.n_neighbors,
                weights=config.weights,
                p=config.p
            )
        else:
            raise ValueError(f"Unsupported model specified in config: {config.model_name}")

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        accuracy = accuracy_score(y_test, y_pred)
        print(f"Run finished. Accuracy: {accuracy:.4f}")
        wandb.log({"accuracy": accuracy, "model_name": config.model_name})

In [8]:
if __name__ == '__main__':
    try:
        X_train, X_test, y_train, y_test = load_and_preprocess_data('water_potability.csv')
    except FileNotFoundError:
        print("Error: 'water_potability.csv' not found. Please make sure the dataset is in the same directory.")
        exit()

    sweep_config_str = """
method: bayes
metric:
  name: accuracy
  goal: maximize
parameters:
  model_name:
    values: ['svm', 'random_forest', 'xgboost', 'knn']

  # --- SVM Parameters ---
  C:
    distribution: log_uniform_values
    min: 0.1
    max: 10.0
  kernel:
    values: ['rbf', 'poly', 'sigmoid']
  gamma:
    distribution: log_uniform_values
    min: 0.001
    max: 1.0
  degree:
    distribution: int_uniform
    min: 2
    max: 5

  # --- Random Forest Parameters (note the '_rf' suffix) ---
  n_estimators_rf:
    distribution: int_uniform
    min: 50
    max: 250
  max_depth_rf:
    distribution: int_uniform
    min: 5
    max: 30
  min_samples_leaf:
    distribution: int_uniform
    min: 1
    max: 4
  criterion:
    values: ['gini', 'entropy']

  # --- XGBoost Parameters (note the '_xgb' suffix) ---
  n_estimators_xgb:
    distribution: int_uniform
    min: 50
    max: 250
  max_depth_xgb:
    distribution: int_uniform
    min: 3
    max: 10
  learning_rate:
    distribution: log_uniform_values
    min: 0.01
    max: 0.3
  subsample:
    distribution: uniform
    min: 0.7
    max: 1.0

  # --- K-Nearest Neighbors Parameters ---
  n_neighbors:
    distribution: int_uniform
    min: 3
    max: 20
  weights:
    values: ['uniform', 'distance']
  p:
    values: [1, 2] # 1 for Manhattan distance, 2 for Euclidean
"""
    sweep_config = yaml.safe_load(sweep_config_str)

    print("Initializing wandb sweep for multiple models...")
    try:
        wandb.login()
        sweep_id = wandb.sweep(sweep=sweep_config, project="waterpredict")
        print(f"Sweep initialized. Sweep ID: {sweep_id}")
    except Exception as e:
        print(f"Could not initialize wandb sweep. Have you logged in with `wandb login`?")
        print(f"Error: {e}")
        exit()

    run_count = 30
    print(f"Starting wandb agent to run {run_count} experiments...")
    wandb.agent(sweep_id, function=train, count=run_count)
    print("\nSweep finished. Check your wandb project dashboard to see the results!")

Loading and preprocessing data...
Data ready.
Initializing wandb sweep for multiple models...


/tmp/ipython-input-2162687033.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['ph'].fillna(df['ph'].mean(), inplace=True)
/tmp/ipython-input-2162687033.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try usin

<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

 ··········


...

accuracy,▁
accuracy,0.62805
model_name,knn


wandb: Agent Starting Run: ueefczow with config:
wandb: 	C: 6.771133181799374
wandb: 	criterion: gini
wandb: 	degree: 4
wandb: 	gamma: 0.1298062257548368
wandb: 	kernel: sigmoid
wandb: 	learning_rate: 0.0246729880745016
wandb: 	max_depth_rf: 9
wandb: 	max_depth_xgb: 9
wandb: 	min_samples_leaf: 4
wandb: 	model_name: xgboost
wandb: 	n_estimators_rf: 146
wandb: 	n_estimators_xgb: 65
wandb: 	n_neighbors: 7
wandb: 	p: 2
wandb: 	subsample: 0.7338567970909112
wandb: 	weights: distance


Starting a new run for model: xgboost with parameters: {'C': 6.771133181799374, 'criterion': 'gini', 'degree': 4, 'gamma': 0.1298062257548368, 'kernel': 'sigmoid', 'learning_rate': 0.0246729880745016, 'max_depth_rf': 9, 'max_depth_xgb': 9, 'min_samples_leaf': 4, 'model_name': 'xgboost', 'n_estimators_rf': 146, 'n_estimators_xgb': 65, 'n_neighbors': 7, 'p': 2, 'subsample': 0.7338567970909112, 'weights': 'distance'}
Run finished. Accuracy: 0.6723


accuracy,▁
accuracy,0.67226
model_name,xgboost



Sweep finished. Check your wandb project dashboard to see the results!
